# LLM Exposure Index – Datenerhebung und Aufbereitung
**Masterarbeit**
Autor: Ayumi Nojima | April 2026

---
## Zusammenfassung – Outputs dieses Notebooks

| Datei | Inhalt | Kapitel |
|-------|--------|---------|
| `processed/onet_pivot.csv` | Bereinigte O\*NET-Fähigkeitsmatrix (normiert, w_ij) | 5.1 |
| `processed/ch_isco_clean.csv` | CH-ISCO-19 HG 1–3 (harmonisiert) | 5.2 |
| `processed/bfs_clean.csv` | BFS ΔBFS_j 2022→2024 inkl. Ausreisser-Flag | 5.3 |
| `processed/bfs_panel_2021_2024.csv` | **NEU:** Panel-BFS 2021–2024 (3 Jahresschritte) | 5.3 |
| `processed/stage1_mapping.csv` | Mapping Stufe 1 (ESCO-Crosswalk) | 5.4 |
| `processed/stage3_manual_review.csv` | Export für manuelle Kodierung | 5.4 |
| `processed/onet_chisco_mapping.csv` | Vollständiges Mapping (alle Stufen) | 5.4 |

**Änderungen gegenüber Vorversion:**
- Pfade in O\*NET-Ladefunktion dynamisch (kein Hardcoding)
- BFS-Panel 2021–2024 zusätzlich aufgebaut (für Fixed-Effects Panel-Analyse)


In [1]:
from pathlib import Path

# Repo-Root = zwei Ebenen über diesem Notebook
REPO_ROOT = Path(__file__).resolve().parents[1] if '__file__' in dir() else Path.cwd().parent

RAW_ONET  = REPO_ROOT / "data" / "raw" / "onet"
RAW_OTHER = REPO_ROOT / "data" / "raw"
PROCESSED = REPO_ROOT / "data" / "processed"
OUTPUT    = REPO_ROOT / "data" / "output"

for p in [PROCESSED, OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

print("Root:", REPO_ROOT)
print("Processed:", PROCESSED)

Root: /workspaces/Master_Thesis
Processed: /workspaces/Master_Thesis/data/processed


## 0. Imports und Konfiguration

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import pearsonr
from rapidfuzz import fuzz, process
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")

# ── Repo-Root robust bestimmen (funktioniert aus notebooks/ und root) ───────
_cwd = Path.cwd()
REPO_ROOT = _cwd.parent if (_cwd / "..").resolve().joinpath("data").exists() else _cwd
if not (REPO_ROOT / "data").exists():
    REPO_ROOT = Path.cwd()
print("Repo-Root:", REPO_ROOT)

RAW_ONET  = REPO_ROOT / "data" / "raw" / "onet"
RAW_OTHER = REPO_ROOT / "data" / "raw"
PROCESSED = REPO_ROOT / "data" / "processed"
OUTPUT    = REPO_ROOT / "data" / "output"
for p in [PROCESSED, OUTPUT]:
    p.mkdir(parents=True, exist_ok=True)

# ── Parameter ──────────────────────────────────────────────────────────────
MISSING_VALUE_THRESHOLD = 0.10   # Max. 10% fehlende Werte je Beruf
MIN_EMPLOYMENT          = 50     # Mindestbeschäftigte 2022 (BFS, in Personen)
SKILL_OVERLAP_THRESHOLD = 0.70   # s_ij-Mindestwert für Indexeinschluss
FUZZY_SCORE_MIN         = 80     # Mindestscore für Fuzzy-Matching (0–100)
MAIN_GROUPS             = [1, 2, 3]  # Wissensintensive Berufshauptgruppen
BFS_BASE_YEAR           = 2022
BFS_TARGET_YEAR         = 2024

print("Konfiguration geladen ✓")


Repo-Root: /workspaces/Master_Thesis
Konfiguration geladen ✓


---
## 5.1 O\*NET-Fähigkeitsprofile

Die O\*NET 30.2-Daten werden aus vier inhaltlichen Teildatensätzen geladen:  
**Skills, Abilities, Work Activities, Knowledge**.  
Ausschliesslich die **Importance-Dimension (Scale ID = "IM")** wird verwendet  
(Eloundou et al., 2023; Kuprecht, 2025).

**Methodische Anmerkung:** Um das Duplikat-Problem bei gleichnamigen Dimensionen  
(z.B. "Mathematics" in Skills und Knowledge) zu vermeiden, wird die Quelle  
als Präfix in den Element-Namen eingebettet (Ansatz in Anlehnung an Fromm, 2022).


In [3]:
def load_onet_file(filename: str, source_label: str) -> pd.DataFrame:
    """Lädt eine O*NET-Datei, filtert auf Scale ID = 'IM' und prefixiert Element-Namen."""
    df = pd.read_excel(RAW_ONET / filename)
    df.columns = df.columns.str.strip()
    df = df[df["Scale ID"] == "IM"].copy()
    df = df[["O*NET-SOC Code", "Title", "Element ID", "Element Name", "Data Value"]].copy()
    df.rename(columns={
        "O*NET-SOC Code": "soc_code",
        "Title":          "soc_title",
        "Element ID":     "element_id",
        "Element Name":   "element_name",
        "Data Value":     "importance"
    }, inplace=True)
    df["element_name"] = source_label + " – " + df["element_name"]
    df["source"] = source_label
    return df

# ── Pfade dynamisch (kein Hardcoding) ─────────────────────────────────────
skills    = load_onet_file("Skills.xlsx",         "Skills")
abilities = load_onet_file("Abilities.xlsx",       "Abilities")
work_act  = load_onet_file("Work Activities.xlsx", "Work Activities")
knowledge = load_onet_file("Knowledge.xlsx",       "Knowledge")

onet_raw = pd.concat([skills, abilities, work_act, knowledge], ignore_index=True)

print(f"O*NET Rohdaten:  {len(onet_raw):>8,} Zeilen")
print(f"  Berufe:        {onet_raw['soc_code'].nunique():>8}")
print(f"  Dimensionen:   {onet_raw['element_name'].nunique():>8}")
print(f"  Quellen:       {onet_raw['source'].unique().tolist()}")


O*NET Rohdaten:   143,934 Zeilen
  Berufe:             894
  Dimensionen:        161
  Quellen:       ['Skills', 'Abilities', 'Work Activities', 'Knowledge']


### Bereinigung der O\*NET-Daten

In [4]:
# ── Schritt 1: Fehlwerte je Beruf prüfen ──────────────────────────────────
total_dims = onet_raw["element_name"].nunique()

missing_by_occ = (
    onet_raw.groupby("soc_code")["importance"]
    .apply(lambda x: x.isna().sum() / total_dims)
    .reset_index(name="missing_rate")
)
excluded_missing = missing_by_occ[missing_by_occ["missing_rate"] > MISSING_VALUE_THRESHOLD]
print(f"Ausgeschlossen (>{int(MISSING_VALUE_THRESHOLD*100)}% fehlende Werte): {len(excluded_missing)} Berufe")

valid_soc  = missing_by_occ[missing_by_occ["missing_rate"] <= MISSING_VALUE_THRESHOLD]["soc_code"]
onet_clean = onet_raw[onet_raw["soc_code"].isin(valid_soc)].copy()

# ── Schritt 2: Fehlwerte mit Dimensions-Median imputen ────────────────────
onet_clean["importance"] = (
    onet_clean["importance"]
    .fillna(onet_clean.groupby("element_name")["importance"].transform("median"))
)

# ── Schritt 3: Min-Max-Normalisierung je Dimension auf [0, 1] (= w_ij) ───
# Vgl. Kuprecht (2025): Normalisierung auf Importance-Skala für Vergleichbarkeit
onet_clean["w_ij"] = onet_clean.groupby("element_name")["importance"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9)
)

print(f"O*NET nach Bereinigung: {onet_clean['soc_code'].nunique()} Berufe | "
      f"{onet_clean['element_name'].nunique()} Dimensionen")

# ── Schritt 4: Pivot → Berufs-Fähigkeits-Matrix ───────────────────────────
onet_pivot = (
    onet_clean
    .pivot_table(index="soc_code", columns="element_name", values="w_ij", aggfunc="mean")
    .reset_index()
)

print(f"Pivot-Matrix: {onet_pivot.shape[0]} Berufe × {onet_pivot.shape[1]-1} Fähigkeitsdimensionen")
onet_pivot.to_csv(PROCESSED / "onet_pivot.csv", index=False)
print("Gespeichert → data/processed/onet_pivot.csv ✓")


Ausgeschlossen (>10% fehlende Werte): 0 Berufe
O*NET nach Bereinigung: 894 Berufe | 161 Dimensionen
Pivot-Matrix: 894 Berufe × 161 Fähigkeitsdimensionen
Gespeichert → data/processed/onet_pivot.csv ✓


---
## 5.2 CH-ISCO-19-Klassifikation

Die CH-ISCO-19-Klassifikation wird vom Bundesamt für Statistik als  
Referenztabelle verwendet. Nur Hauptgruppen 1–3 (wissensintensive Berufe).  
Vorgehen analog Kuprecht (2025): 4-stellige Codes aus Spalte 3, Hauptgruppe via ffill aus Spalte 0.


In [5]:
def extract_chisco(sheet_name: str = "2022") -> pd.DataFrame:
    """
    Extrahiert 4-stellige CH-ISCO-19-Codes und Bezeichnungen für HG 1–3
    aus CH_ISCO19.xlsx. Hauptgruppe wird via ffill aus Spalte 0 abgeleitet.
    """
    df = pd.read_excel(RAW_OTHER / "CH_ISCO19.xlsx", sheet_name=sheet_name, header=None)
    df["hg"] = pd.to_numeric(df[0], errors="coerce").ffill()

    col3_str = df[3].astype(str)
    mask = col3_str.str.match(r"^\d{4}$") & df["hg"].isin([float(g) for g in MAIN_GROUPS])

    result = df[mask][[3, 5, "hg"]].copy()
    result.columns = ["ch_isco_4digit", "ch_isco_label", "main_group"]
    result["ch_isco_4digit"] = result["ch_isco_4digit"].astype(str)
    result["main_group"]     = result["main_group"].astype(int)

    # Bezeichnungen harmonisieren für Fuzzy-Matching (Stufe 2)
    result["label_clean"] = (
        result["ch_isco_label"].astype(str)
        .str.strip().str.lower()
        .str.replace(r"[äöü]", lambda m: {"ä":"ae","ö":"oe","ü":"ue"}[m.group()], regex=True)
        .str.replace(r"[^a-z0-9 ]", " ", regex=True)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    return result.reset_index(drop=True)

ch_isco = extract_chisco("2022")
print(f"CH-ISCO-19 Hauptgruppen 1–3: {len(ch_isco)} Berufsklassen")
print(ch_isco["main_group"].value_counts().sort_index().to_string())
ch_isco.to_csv(PROCESSED / "ch_isco_clean.csv", index=False)
print("\nGespeichert → data/processed/ch_isco_clean.csv ✓")
ch_isco.head(3)


CH-ISCO-19 Hauptgruppen 1–3: 271 Berufsklassen
main_group
1     44
2    119
3    108

Gespeichert → data/processed/ch_isco_clean.csv ✓


,ch_isco_4digit,ch_isco_label,main_group,label_clean
0,1000,"Führungskräfte, onA",1,fuehrungskraefte ona
1,1100,"Geschäftsführer, Vorstände, leitende Verwaltun...",1,geschaeftsfuehrer vorstaende leitende verwaltu...
2,1110,Angehörige gesetzgebender Körperschaften und l...,1,angehoerige gesetzgebender koerperschaften und...


---
## 5.3 BFS-Strukturerhebung (ΔBFS_j)

Beschäftigungsdaten für 2022 und 2024 aus separaten Jahres-Sheets.  
**Wichtig:** Die gepoolten Sheets (z.B. "2022-2024") enthalten 3-Jahres-Durchschnitte  
und sind nicht für die ΔBFS-Berechnung geeignet — nur Einzeljahres-Sheets verwenden.

$$\Delta BFS_j = \frac{Beschäftigte_{2024} - Beschäftigte_{2022}}{Beschäftigte_{2022}} \times 100$$


In [6]:
def extract_bfs_year(sheet_name: str, year_label: str) -> pd.DataFrame:
    """
    Extrahiert Beschäftigtenzahlen (Total, in 1000) für ein einzelnes Jahr
    aus CH_ISCO19.xlsx für Hauptgruppen 1–3, 4-stellige Codes.
    Supprimierte Werte ('X') → NaN.
    """
    df = pd.read_excel(REPO_ROOT / "data" / "raw" / "CH_ISCO19.xlsx",
                       sheet_name=sheet_name, header=None)
    df["hg"] = pd.to_numeric(df[0], errors="coerce").ffill()
    col3_str = df[3].astype(str)
    mask = col3_str.str.match(r"^\d{4}$") & df["hg"].isin([float(g) for g in MAIN_GROUPS])
    result = df[mask][[3, 6]].copy()
    result.columns = ["ch_isco_4digit", year_label]
    result["ch_isco_4digit"] = result["ch_isco_4digit"].astype(str)
    result[year_label] = pd.to_numeric(result[year_label], errors="coerce")
    return result.reset_index(drop=True)

# ── Primäre BFS-Daten: 2022 → 2024 (H3 Hauptanalyse) ─────────────────────
bfs22 = extract_bfs_year(str(BFS_BASE_YEAR),   f"employed_{BFS_BASE_YEAR}")
bfs24 = extract_bfs_year(str(BFS_TARGET_YEAR),  f"employed_{BFS_TARGET_YEAR}")

bfs = bfs22.merge(bfs24, on="ch_isco_4digit", how="inner")
MIN_EMP_SCALED = MIN_EMPLOYMENT / 1000
n_before = len(bfs)
bfs = bfs.dropna(subset=[f"employed_{BFS_BASE_YEAR}", f"employed_{BFS_TARGET_YEAR}"])
bfs = bfs[bfs[f"employed_{BFS_BASE_YEAR}"] >= MIN_EMP_SCALED].copy()
print(f"Ausgeschlossen (<{MIN_EMPLOYMENT} Beschäftigte oder NaN): {n_before - len(bfs)} Berufsgruppen")

bfs["delta_bfs"] = (
    (bfs[f"employed_{BFS_TARGET_YEAR}"] - bfs[f"employed_{BFS_BASE_YEAR}"])
    / bfs[f"employed_{BFS_BASE_YEAR}"] * 100
)
Q1, Q3 = bfs["delta_bfs"].quantile([0.25, 0.75])
IQR     = Q3 - Q1
bfs["is_outlier"] = ~bfs["delta_bfs"].between(Q1 - 1.5*IQR, Q3 + 1.5*IQR)

print(f"BFS-Datensatz (2022→2024): {len(bfs)} Berufsgruppen | Ausreisser: {bfs['is_outlier'].sum()}")
print(f"ΔBFS: Mean={bfs['delta_bfs'].mean():.1f}% | Std={bfs['delta_bfs'].std():.1f}%")
bfs.to_csv(PROCESSED / "bfs_clean.csv", index=False)
print("Gespeichert → data/processed/bfs_clean.csv ✓")

# ── Panel-BFS: 2021–2024 (Prio 2: Fixed-Effects Panel-Analyse) ────────────
# Drei Jahresschritte: 2021→2022 (Vor-KI), 2022→2023, 2023→2024 (Nach-KI)
# Covid-Periode 2019→2021 bewusst ausgeschlossen (Std der ΔBFS 2019→2020 = 44%)
PANEL_YEARS = ["2021", "2022", "2023", "2024"]

bfs_panel_dfs = {}
for y in PANEL_YEARS:
    bfs_panel_dfs[y] = extract_bfs_year(y, f"emp_{y}")

bfs_panel = bfs_panel_dfs["2021"]
for y in PANEL_YEARS[1:]:
    bfs_panel = bfs_panel.merge(bfs_panel_dfs[y], on="ch_isco_4digit", how="inner")

# Mindestfallzahl: 50 Beschäftigte in 2021
bfs_panel = bfs_panel.dropna()
bfs_panel = bfs_panel[bfs_panel["emp_2021"] >= MIN_EMP_SCALED].copy()

# Wachstumsraten je Jahresschritt
for y1, y2 in [("2021", "2022"), ("2022", "2023"), ("2023", "2024")]:
    bfs_panel[f"delta_{y1}_{y2}"] = (
        (bfs_panel[f"emp_{y2}"] - bfs_panel[f"emp_{y1}"])
        / bfs_panel[f"emp_{y1}"] * 100
    )

# Post-KI Indikator: ChatGPT eingeführt Nov. 2022
# 2021→2022 = Vor-KI (post=0), 2022→2023 und 2023→2024 = Nach-KI (post=1)
print(f"\nPanel-BFS (2021–2024): {len(bfs_panel)} Berufsgruppen (alle 4 Jahre vollständig)")
for y1, y2 in [("2021","2022"),("2022","2023"),("2023","2024")]:
    d = bfs_panel[f"delta_{y1}_{y2}"]
    print(f"  ΔBFS {y1}→{y2}: Mean={d.mean():.1f}% | Std={d.std():.1f}%")

bfs_panel.to_csv(PROCESSED / "bfs_panel_2021_2024.csv", index=False)
print("Gespeichert → data/processed/bfs_panel_2021_2024.csv ✓")


Ausgeschlossen (<50 Beschäftigte oder NaN): 56 Berufsgruppen
BFS-Datensatz (2022→2024): 215 Berufsgruppen | Ausreisser: 15
ΔBFS: Mean=6.6% | Std=23.5%
Gespeichert → data/processed/bfs_clean.csv ✓

Panel-BFS (2021–2024): 214 Berufsgruppen (alle 4 Jahre vollständig)
  ΔBFS 2021→2022: Mean=3.1% | Std=17.7%
  ΔBFS 2022→2023: Mean=5.6% | Std=22.4%
  ΔBFS 2023→2024: Mean=2.0% | Std=16.6%
Gespeichert → data/processed/bfs_panel_2021_2024.csv ✓


### Ausgangslage vor Mapping

In [7]:
print("=" * 55)
print("AUSGANGSLAGE VOR MAPPING")
print("=" * 55)
print(f"  O*NET-Berufe (bereinigt):        {onet_pivot.shape[0]:>5}")
print(f"  CH-ISCO-19 Klassen (HG 1–3):     {len(ch_isco):>5}")
print(f"  BFS-Berufsgruppen (bereinigt):   {len(bfs):>5}")
print(f"  Ziel-Sample:                      ~100")
print("=" * 55)


AUSGANGSLAGE VOR MAPPING
  O*NET-Berufe (bereinigt):          894
  CH-ISCO-19 Klassen (HG 1–3):       271
  BFS-Berufsgruppen (bereinigt):     215
  Ziel-Sample:                      ~100


---
## 5.4 Mapping O\*NET → CH-ISCO-19 (dreistufig)

Das Mapping verbindet amerikanische O\*NET-SOC-Codes mit Schweizer CH-ISCO-19-Klassen.  
**Dreistufiges Verfahren** (vgl. Kuprecht, 2025; Fromm, 2022):

| Stufe | Methode | Grundlage |
|-------|---------|-----------|
| 1 | ESCO-Crosswalk (ISCO-08 als Brücke) | ESCO→O\*NET-SOC Datei von onetcenter.org |
| 2 | Analyse nicht gemappter Codes | Kategorisierung in HG 4–9 (korrekt ausgeschlossen) vs. fehlend |
| 3 | Manuelle Korrektur | Nur Codes die gar nicht im Crosswalk vorhanden sind |

### Stufe 1: ESCO-Crosswalk laden


In [8]:
# ESCO→O*NET-SOC Crosswalk (heruntergeladen von https://www.onetcenter.org/crosswalks.html)
# Datei: data/raw/ESCO_to_ONET-SOC.xlsx

esco_raw = pd.read_excel(REPO_ROOT / "data" / "raw" / "ESCO_to_ONET-SOC.xlsx", sheet_name="O-NET-SOC 2019 Crosswalks", header=2)
esco_raw.columns = esco_raw.iloc[0].tolist()
esco_raw = esco_raw.iloc[1:].reset_index(drop=True)
esco_raw.columns = [str(c).strip() for c in esco_raw.columns]

# ISCO-4-digit aus ESCO/ISCO Code extrahieren (Format: "1234.56" → "1234")
esco_raw["isco_4digit"] = esco_raw["ESCO/ISCO Code"].astype(str).str.extract(r"^(\d{4})")
esco_raw["isco_hg"]     = pd.to_numeric(esco_raw["isco_4digit"].str[:1], errors="coerce")

print(f"ESCO Crosswalk geladen: {len(esco_raw):,} Einträge")
print(f"ISCO Hauptgruppen vorhanden: {sorted(esco_raw['isco_hg'].dropna().unique().astype(int))}")
print(f"Einträge HG 1–3: {esco_raw['isco_hg'].isin([1,2,3]).sum():,}")


ESCO Crosswalk geladen: 8,629 Einträge
ISCO Hauptgruppen vorhanden: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9)]
Einträge HG 1–3: 5,359


### Stufe 1: Hierarchisches Matching

In [9]:
# Nur HG 1–3 aus ESCO-Crosswalk
hg123 = esco_raw[esco_raw["isco_hg"].isin([1.0, 2.0, 3.0])].copy()

# Eindeutiges 1:1 Mapping: häufigsten ISCO-4-digit Code je O*NET-SOC wählen
# Bei Gleichstand: niedrigerer ISCO-Code bevorzugt (deterministisch)
# Methodisch: Aggregation auf ISCO-Ebene analog Kuprecht (2025)
best_map = (
    hg123.groupby(["O*NET-SOC 2019 Code", "isco_4digit"])
    .size()
    .reset_index(name="count")
    .sort_values(["count", "isco_4digit"], ascending=[False, True])
    .drop_duplicates("O*NET-SOC 2019 Code")
    [["O*NET-SOC 2019 Code", "isco_4digit"]]
    .rename(columns={"O*NET-SOC 2019 Code": "soc_code"})
)

stage1 = onet_pivot[["soc_code"]].merge(best_map, on="soc_code", how="left")
mapped_s1   = stage1.dropna(subset=["isco_4digit"]).copy()
unmapped_s1 = stage1[stage1["isco_4digit"].isna()]["soc_code"].values

print(f"Stufe 1 – Hierarchisches Matching via ESCO-Crosswalk:")
print(f"  Zugeordnet:    {len(mapped_s1):>4} Berufe ({len(mapped_s1)/len(onet_pivot)*100:.1f}%)")
print(f"  Nicht gemappt: {len(unmapped_s1):>4} Berufe")

mapped_s1.to_csv(PROCESSED / "stage1_mapping.csv", index=False)
print("\nGespeichert → data/processed/stage1_mapping.csv ✓")


Stufe 1 – Hierarchisches Matching via ESCO-Crosswalk:
  Zugeordnet:     714 Berufe (79.9%)
  Nicht gemappt:  180 Berufe

Gespeichert → data/processed/stage1_mapping.csv ✓


### Stufe 2: Analyse nicht gemappter Codes

**Methodische Erkenntnis:** Der Grossteil der nicht gemappten O\*NET-Codes ist  
im ESCO-Crosswalk vorhanden, aber mit ISCO HG 4–9 verknüpft — also Berufe, die  
per Definition nicht zu den wissensintensiven Gruppen 1–3 gehören.  
Diese werden korrekt ausgeschlossen (kein methodischer Fehler).


In [10]:
# Analyse der nicht gemappten Codes
all_esco_check = esco_raw[["O*NET-SOC 2019 Code", "isco_4digit", "isco_hg"]].rename(
    columns={"O*NET-SOC 2019 Code": "soc_code"}
)
check = pd.DataFrame({"soc_code": unmapped_s1}).merge(all_esco_check, on="soc_code", how="left")

cat_a = check[check["isco_hg"].notna()]["soc_code"].nunique()   # In ESCO, aber HG 4–9
cat_b = check[check["isco_hg"].isna()]["soc_code"].unique()      # Gar nicht in ESCO

print(f"Stufe 2 – Analyse nicht gemappter Berufe:")
print(f"  Kategorie A: In ESCO vorhanden, ISCO HG 4–9 → korrekt ausgeschlossen: {cat_a} Codes")
print(f"  Kategorie B: Gar nicht im ESCO-Crosswalk → Stufe 3:                    {len(cat_b)} Codes")

# Nur Kategorie B geht weiter
unmapped_s2 = cat_b
mapped_s2   = pd.DataFrame(columns=["soc_code", "isco_4digit", "mapping_stage"])
print(f"\nStufe 2 produziert keine automatischen Matches.")
print(f"Begründung: Cross-language Fuzzy-Matching (EN↔DE) liefert keine validen Treffer.")
print(f"Alle {len(cat_b)} Kategorie-B-Codes gehen in manuelle Prüfung (Stufe 3).")


Stufe 2 – Analyse nicht gemappter Berufe:
  Kategorie A: In ESCO vorhanden, ISCO HG 4–9 → korrekt ausgeschlossen: 179 Codes
  Kategorie B: Gar nicht im ESCO-Crosswalk → Stufe 3:                    4 Codes

Stufe 2 produziert keine automatischen Matches.
Begründung: Cross-language Fuzzy-Matching (EN↔DE) liefert keine validen Treffer.
Alle 4 Kategorie-B-Codes gehen in manuelle Prüfung (Stufe 3).


### Stufe 3: Manuelle Korrektur

In [11]:
# Occupation Data für Titel der manuell zu prüfenden Berufe
occ_data = pd.read_excel(RAW_ONET / "Occupation Data.xlsx")
occ_data.columns = occ_data.columns.str.strip()
occ_data.rename(columns={"O*NET-SOC Code": "soc_code", "Title": "onet_title"}, inplace=True)
occ_data = occ_data[["soc_code", "onet_title"]].copy()

manual_review = occ_data[occ_data["soc_code"].isin(unmapped_s2)].copy()
manual_review["isco_4digit_coder1"] = ""
manual_review["isco_4digit_coder2"] = ""
manual_review["isco_4digit_final"]  = ""

manual_path = PROCESSED / "stage3_manual_review.csv"
manual_review.to_csv(manual_path, index=False)
print(f"Stufe 3 – Manuelle Prüfung:")
print(f"  {len(manual_review)} Berufe exportiert → {manual_path}")
print(f"  → Zwei unabhängige Kodierer; Krippendorff-α Ziel: > 0.80")
print(f"  → Als 'stage3_completed.csv' speichern wenn fertig")


Stufe 3 – Manuelle Prüfung:
  4 Berufe exportiert → /workspaces/Master_Thesis/data/processed/stage3_manual_review.csv
  → Zwei unabhängige Kodierer; Krippendorff-α Ziel: > 0.80
  → Als 'stage3_completed.csv' speichern wenn fertig


### Mapping zusammenführen

In [12]:
# Stufe 3 einlesen (nach manueller Kodierung), sonst leer weiterfahren
try:
    mapped_s3 = pd.read_csv(PROCESSED / "stage3_completed.csv")
    mapped_s3 = (mapped_s3[["soc_code", "isco_4digit_final"]]
                 .rename(columns={"isco_4digit_final": "isco_4digit"}))
    mapped_s3["mapping_stage"] = 3
    print(f"Stufe 3 geladen: {len(mapped_s3)} Berufe")
except FileNotFoundError:
    print("HINWEIS: stage3_completed.csv nicht vorhanden – Stufe 3 übersprungen")
    mapped_s3 = pd.DataFrame(columns=["soc_code", "isco_4digit", "mapping_stage"])

# Stufe 1
mapped_s1_df = mapped_s1[["soc_code", "isco_4digit"]].copy()
mapped_s1_df["mapping_stage"] = 1

# Stufe 2 absichern (kann leer sein)
if len(mapped_s2) > 0:
    mapped_s2_df = mapped_s2[["soc_code", "isco_4digit", "mapping_stage"]].copy()
else:
    mapped_s2_df = pd.DataFrame(columns=["soc_code", "isco_4digit", "mapping_stage"])

# Kombinieren — Stufe 1 hat Priorität (drop_duplicates keep='first')
mapping_all = (
    pd.concat([mapped_s1_df, mapped_s2_df, mapped_s3], ignore_index=True)
    .drop_duplicates("soc_code", keep="first")
)

# Nur HG 1–3 behalten
mapping_all = mapping_all[
    mapping_all["isco_4digit"].astype(str).str[:1].isin(["1", "2", "3"])
].copy()

stage_counts = mapping_all["mapping_stage"].value_counts().sort_index()
print("\nMapping-Gesamtergebnis:")
for stage, count in stage_counts.items():
    print(f"  Stufe {int(stage)}: {count:>4} Berufe")
print(f"  Total:  {len(mapping_all):>4} Berufe gemappt")
print(f"  Abdeckung O*NET gesamt: {len(mapping_all)/len(onet_pivot)*100:.1f}%")

mapping_all.to_csv(PROCESSED / "onet_chisco_mapping.csv", index=False)
print("\nGespeichert → data/processed/onet_chisco_mapping.csv ✓")


Stufe 3 geladen: 4 Berufe

Mapping-Gesamtergebnis:
  Stufe 1:  714 Berufe
  Total:   714 Berufe gemappt
  Abdeckung O*NET gesamt: 79.9%

Gespeichert → data/processed/onet_chisco_mapping.csv ✓


### Hold-out-Validierung (20%)

20% der gemappten Berufe werden als Hold-out-Set zurückgehalten,  
um die Generalisierbarkeit des Mapping-Verfahrens zu validieren (Hastie et al., 2009).


In [13]:
train_mapping, holdout_mapping = train_test_split(
    mapping_all, test_size=0.20, random_state=42
)
holdout_mapping.to_csv(PROCESSED / "holdout_mapping.csv",  index=False)
train_mapping.to_csv(PROCESSED  / "train_mapping.csv",   index=False)

print(f"Hold-out-Set:   {len(holdout_mapping)} Berufe ({len(holdout_mapping)/len(mapping_all)*100:.0f}%)")
print(f"Training-Set:   {len(train_mapping)} Berufe ({len(train_mapping)/len(mapping_all)*100:.0f}%)")


Hold-out-Set:   143 Berufe (20%)
Training-Set:   571 Berufe (80%)
